# Beating the default compiler, measured

| | |
|---|---|
| **Level** | Advanced |
| **Time** | 60 to 90 minutes |
| **Prerequisites** | Transpilation stages, qubit routing; QAOA helps |
| **Default devices** | Rigetti Cepheus and IQM Garnet |
| **Hardware jobs** | 2 per device |
| **Approximate cost** | about 640 credits on Garnet at 2000 shots; Rigetti is billed by execution time, about 10 credits per job in our tests |
| **Hardware notes** | Tested 25 September 2026 at 100 shots. On IQM Garnet, where our own compilations ran exactly as written (56 and 44 two-qubit gates), the swept circuit kept 66% of the ideal advantage against 53% for the default. On Rigetti Cepheus the platform recompiled both orderings to over 100 two-qubit gates and both came out at random guessing. |

Shot counts for the hardware runs are set at the end of the **Setup** cell. The devices are set in the first hardware cell. Credits are charged only when a hardware cell runs.

*QUEST intermediate and advanced series: Systems, Hardware and Engineering*

`transpile(circuit, backend)` is a single line, but it makes most of the decisions that determine whether a circuit returns a result or noise. It chooses which physical qubits hold your logical qubits, inserts SWAP gates to satisfy the device's connections, rewrites your gates into the device's native set, and optimises what it can. When two-qubit gates carry about a percent of error each, a good compilation and a poor one can be the difference between an answer and a flat distribution.

The Grover notebook in this series printed the transpiled depth and two-qubit gate count for each device. This notebook asks whether that count is the best available or just the first one the compiler produced.

It is usually not the best available, for two reasons. The layout and routing passes are randomised, so a different seed gives a different circuit, and the spread is wide enough to be worth sampling. And the compiler must preserve the circuit you gave it, so it will not reorder commuting gates even when that would reduce routing. Both gaps can be closed with free classical computation.

We close them on a QAOA Max-Cut layer, measure the gate counts, and then check what matters: whether the smaller circuit gives a better answer on real hardware.

**Learning objectives**

1. Name the stages of the transpiler and say which are randomised.
2. Measure the spread in compiled circuit size across seeds, and use the best of $N$.
3. Recognise commuting gates and verify that reordering them preserves the unitary.
4. Show that no fixed reordering rule is always best, and sweep instead.
5. Compare compilations by the answers they produce on hardware, as well as by gate count.
6. Say when extra compilation effort stops paying off.

**Background needed:** QAOA at the level of the Max-Cut notebook in this series (restated briefly here), and two-qubit gate count as a rough measure of error. The benchmarking notebook in this series is a useful companion.


## What the one-liner does

Qiskit's transpiler runs six stages in order.

| Stage | Job | Randomised |
|---|---|---|
| `init` | Unroll to a workable form, run early simplifications | no |
| `layout` | Assign logical qubits to physical qubits | yes, at levels 1 and above |
| `routing` | Insert SWAPs so every two-qubit gate acts on connected qubits | yes |
| `translation` | Rewrite into the device's native basis | no |
| `optimization` | Cancel and merge gates, resynthesise blocks | partly |
| `scheduling` | Place gates in time, insert delays | no |

Layout and routing are where the cost is decided, and they are the two stages that are randomised. Both are NP-hard in general, so Qiskit uses heuristics seeded from `seed_transpiler`. Change the seed and you get a different, equally valid circuit with a different gate count. Nothing in the interface suggests you should look at more than one.

The optimisation level selects how much work goes into these stages: level 0 is essentially none, level 1 is light, level 2 adds more optimisation passes, and level 3 adds heavier resynthesis. Higher is not automatically better for the quantity you care about, and we will measure rather than assume.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import time

from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

from qbraid.runtime import QbraidProvider

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 110

BASIS = ['cx', 'rz', 'sx', 'x']       # a generic superconducting native set
rng = np.random.default_rng(2)

print("Setup complete.")

# Shot counts for the hardware runs. More shots reduce statistical error but cost
# more on most devices; the README lists prices.
HW_SHOTS = 2000
QUEST_JOB_TAGS = {"quest": "noise-compiler"}   # labels this notebook's hardware jobs for QUEST usage statistics


## The test case

A QAOA Max-Cut layer is a good subject for this. Its structure is a run of two-qubit $ZZ$ rotations, one per graph edge, and those rotations all commute with each other, so their order is ours to choose. It also has an unambiguous figure of merit: the expected cut value of the measured bitstrings, which we can compare against both the ideal circuit and against random guessing.

We take an 8-node random graph and a device modelled as a line of 8 qubits. The line is a stand-in. The platform does not publish coupling maps, so we compile against a plausible worst case and let the vendor's own compiler do the real placement when we submit; what we control is which of several equivalent circuits we hand it.

In [ ]:
N_NODES, N_EDGES = 8, 14
G = nx.gnm_random_graph(N_NODES, N_EDGES, seed=2)
EDGES = list(G.edges())
CMAP = CouplingMap.from_line(N_NODES)

# Every cut value, for scoring and for the angle search.
CUTS = np.array([sum(1 for u, v in EDGES if (k >> u & 1) != (k >> v & 1))
                 for k in range(2 ** N_NODES)])
RANDOM_GUESS = N_EDGES / 2


def qaoa_layer(order, gamma, beta):
    """One QAOA layer. `order` fixes the sequence of the commuting ZZ terms."""
    qc = QuantumCircuit(N_NODES)
    qc.h(range(N_NODES))
    for u, v in order:
        qc.cx(u, v)
        qc.rz(2 * gamma, v)
        qc.cx(u, v)
    qc.rx(2 * beta, range(N_NODES))
    return qc


print(f"graph: {N_NODES} nodes, {N_EDGES} edges")
print(f"maximum cut: {CUTS.max()}   random guessing: {RANDOM_GUESS:.1f}")
nx.draw_circular(G, with_labels=True, node_color='#a02580', font_color='white',
                 node_size=650, edge_color='#888888')
plt.show()

In [ ]:
# Grid search for the best p=1 angles, so that the circuit has signal worth preserving.
best_val, best_angles = -1.0, None
for g in np.linspace(0, np.pi, 41):
    for b in np.linspace(0, np.pi / 2, 21):
        probs = np.abs(Statevector(qaoa_layer(EDGES, g, b)).data) ** 2
        val = float(probs @ CUTS)
        if val > best_val:
            best_val, best_angles = val, (g, b)

GAMMA, BETA = best_angles
IDEAL_CUT = best_val
print(f"best p=1 angles: gamma = {GAMMA:.4f}, beta = {BETA:.4f}")
print(f"ideal expected cut: {IDEAL_CUT:.3f}  (random guessing {RANDOM_GUESS:.1f}, "
      f"maximum {CUTS.max()})")
print(f"advantage available over random guessing: {IDEAL_CUT - RANDOM_GUESS:.3f}")

The advantage over random guessing is what noise will eat, and it is what a better compilation has to protect. Reporting an absolute cut value hides this: a completely decohered device still returns about $m/2$, so a number near 7 means nothing on its own. Every result below is quoted as the fraction of that advantage retained,

$$\text{retained} = \frac{\langle C \rangle_{\text{measured}} - m/2}{\langle C \rangle_{\text{ideal}} - m/2}$$

which is 1 for a perfect device and 0 for one that has told you nothing.

## Optimisation levels

The first knob. Each level is a different pass manager; higher levels do more work at compile time.

In [ ]:
def compile_stats(qc, level, seed):
    t = transpile(qc, coupling_map=CMAP, basis_gates=BASIS,
                  optimization_level=level, seed_transpiler=seed)
    return t, t.count_ops().get('cx', 0), t.depth()


base = qaoa_layer(EDGES, GAMMA, BETA)
rows = []
for level in range(4):
    t0 = time.perf_counter()
    _, cx, depth = compile_stats(base, level, 0)
    rows.append({'level': level, 'CX': cx, 'depth': depth,
                 'compile time (s)': round(time.perf_counter() - t0, 3)})

print(f"logical circuit before compilation: "
      f"{base.count_ops().get('cx', 0)} CX, depth {base.depth()}")
pd.DataFrame(rows).set_index('level')

Level 0 shows the raw cost of routing with no cleanup. The jump to level 1 is large, the jump from 1 to 2 is smaller, and 3 usually buys little over 2 on a circuit this shape while costing noticeably more compile time. That last point is worth remembering: level 3 is not a free upgrade, and on larger circuits its resynthesis passes dominate the compile.

## The same call, thirty times

Layout and routing are randomised. Nothing about the API hints at it, and the default `seed_transpiler=None` means consecutive calls in the same session can return different circuits.

In [ ]:
SEEDS = range(30)
by_level = {lvl: [compile_stats(base, lvl, s)[1] for s in SEEDS] for lvl in (1, 2, 3)}

fig, ax = plt.subplots(figsize=(9.5, 5))
colors = {1: '#1a5285', 2: '#a02580', 3: '#2d7a4f'}
bins = np.arange(min(min(v) for v in by_level.values()) - 1,
                 max(max(v) for v in by_level.values()) + 3) - 0.5
for lvl, vals in by_level.items():
    ax.hist(vals, bins=bins, alpha=0.65, color=colors[lvl],
            label=f'level {lvl}: best {min(vals)}, median {int(np.median(vals))}, worst {max(vals)}')
ax.set_xlabel('CX gates after compilation')
ax.set_ylabel('number of seeds')
ax.set_title('The same circuit and the same call, compiled with 30 different seeds')
ax.grid(alpha=0.3, axis='y')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

for lvl, vals in by_level.items():
    print(f"level {lvl}: best {min(vals)}, median {int(np.median(vals))}, worst {max(vals)}  "
          f"-> best-of-30 is {100 * (np.median(vals) - min(vals)) / np.median(vals):.0f}% "
          f"below the median")

A spread of several gates at every level, for identical input. If you call the transpiler once you get a sample from this distribution, and there is no reason to think it is a good one. Taking the best of thirty costs a second of laptop time and is strictly better than taking the first.

Two things in that plot are worth more than the spread itself. The level with the narrowest distribution is not the level with the best circuit, and a pass that reliably returns a mediocre circuit would have no spread at all. And the best single circuit found across all three levels does not have to come from level 3. Optimisation level is one more parameter to sweep, not a dial to turn up.

## The gates the compiler will not reorder

Every $ZZ$ rotation in the phase separator commutes with every other, because they are all diagonal in the computational basis. So the $14!$ orderings of these edges all implement exactly the same unitary. The compiler will not explore them. It is required to preserve the circuit you handed it, and while it does perform limited commutation analysis, searching orderings for routing benefit is not its job.

It is ours, and it is worth doing, because the ordering determines how far the routing pass has to move qubits around.

First, verify the claim rather than asserting it.

In [ ]:
shuffled = list(EDGES)
rng.shuffle(shuffled)
reversed_order = list(reversed(EDGES))
sorted_order = sorted(EDGES, key=lambda e: (min(e), max(e)))

reference = Operator(qaoa_layer(EDGES, GAMMA, BETA))
for label, order in [('reversed', reversed_order), ('sorted', sorted_order),
                     ('shuffled', shuffled)]:
    same = reference.equiv(Operator(qaoa_layer(order, GAMMA, BETA)))
    cx = compile_stats(qaoa_layer(order, GAMMA, BETA), 3, 0)[1]
    print(f"{label:9s} same unitary: {same}    CX after compilation: {cx}")
print(f"{'as given':9s} same unitary: True    CX after compilation: "
      f"{compile_stats(base, 3, 0)[1]}")

Same unitary in every case, verified on the full $2^8 \times 2^8$ operator, and a materially different compiled cost. That gap is not the compiler underperforming. It is information the compiler was never given.

## No fixed heuristic wins

The tempting next step is a clever ordering rule: sort the edges so that consecutive terms act on nearby qubits, keeping the routing local. It is easy to write and it does help, sometimes. It also loses, sometimes, and there is no way to tell which case you are in without compiling.

In [ ]:
def greedy_locality(edges):
    """Order edges so each term starts near where the previous one finished."""
    remaining, out, pos = list(edges), [], 0
    while remaining:
        remaining.sort(key=lambda e: min(abs(e[0] - pos), abs(e[1] - pos)))
        e = remaining.pop(0)
        out.append(e)
        pos = (e[0] + e[1]) // 2
    return out


def best_over_seeds(order, level=3, seeds=range(20)):
    return min(compile_stats(qaoa_layer(order, GAMMA, BETA), level, s)[1] for s in seeds)


rows = []
for nodes, edges_n, seed in [(6, 9, 1), (8, 14, 2), (10, 20, 3), (12, 26, 4), (14, 32, 5)]:
    g = nx.gnm_random_graph(nodes, edges_n, seed=seed)
    e = list(g.edges())
    cm, nn = CouplingMap.from_line(nodes), nodes

    def cx_for(order):
        qc = QuantumCircuit(nn)
        qc.h(range(nn))
        for u, v in order:
            qc.cx(u, v); qc.rz(1.0, v); qc.cx(u, v)
        qc.rx(1.0, range(nn))
        return min(transpile(qc, coupling_map=cm, basis_gates=BASIS,
                             optimization_level=3, seed_transpiler=s).count_ops().get('cx', 0)
                   for s in range(20))

    rows.append({'nodes': nodes, 'edges': edges_n,
                 'as given': cx_for(e),
                 'sorted': cx_for(sorted(e, key=lambda x: (min(x), max(x)))),
                 'greedy': cx_for(greedy_locality(e))})

df = pd.DataFrame(rows).set_index('nodes')
df['best heuristic beats as-given by'] = [
    f"{100 * (r['as given'] - min(r['sorted'], r['greedy'])) / r['as given']:+.0f}%"
    for _, r in df.iterrows()]
df

The greedy rule wins at 8 nodes and loses at 10, by a wide margin in both directions. A rule that is sometimes 15% better and sometimes 15% worse is not a rule you can apply blind, and picking whichever rule looked good on the instance you tested is how you end up reporting a speedup that does not reproduce.

The way out is not a better heuristic. It is to stop guessing and compile several orderings.

## Sweep the orderings and the seeds together

Generate random orderings, compile each with several seeds, keep the smallest result. The whole search is classical, it is embarrassingly parallel, and its cost is seconds against a hardware job that costs credits and queue time.

In [ ]:
def sweep(edges, n_orders=8, n_seeds=6, level=3):
    """Compile many equivalent orderings and return the best circuit found."""
    default = transpile(qaoa_layer(edges, GAMMA, BETA), coupling_map=CMAP,
                        basis_gates=BASIS, optimization_level=level, seed_transpiler=0)
    best_circ, best_cx, best_order = default, default.count_ops().get('cx', 0), list(edges)
    trace = []
    r = np.random.default_rng(7)

    for _ in range(n_orders):
        order = list(edges)
        r.shuffle(order)
        for s in range(n_seeds):
            t = transpile(qaoa_layer(order, GAMMA, BETA), coupling_map=CMAP,
                          basis_gates=BASIS, optimization_level=level, seed_transpiler=s)
            cx = t.count_ops().get('cx', 0)
            trace.append(cx)
            if cx < best_cx:
                best_circ, best_cx, best_order = t, cx, order
    return default, best_circ, best_order, trace


t0 = time.perf_counter()
default_circ, swept_circ, swept_order, trace = sweep(EDGES)
elapsed = time.perf_counter() - t0

print(f"one default call:  {default_circ.count_ops().get('cx', 0)} CX, "
      f"depth {default_circ.depth()}")
print(f"best of {len(trace)} compilations: {swept_circ.count_ops().get('cx', 0)} CX, "
      f"depth {swept_circ.depth()}")
print(f"reduction: {100 * (default_circ.count_ops().get('cx', 0) - swept_circ.count_ops().get('cx', 0)) / default_circ.count_ops().get('cx', 0):.0f}% "
      f"fewer two-qubit gates, for {elapsed:.1f} s of classical compute")

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.5))
running = np.minimum.accumulate(trace)
ax.plot(range(1, len(trace) + 1), trace, 'o', color='#c63792', alpha=0.45,
        markersize=5, label='each compilation')
ax.plot(range(1, len(trace) + 1), running, '-', color='#1a5285', linewidth=2.5,
        label='best so far')
ax.axhline(default_circ.count_ops().get('cx', 0), color='k', linestyle='--',
           linewidth=2, label='single default call')
ax.set_xlabel('compilations tried')
ax.set_ylabel('CX gates')
ax.set_title('Sweeping equivalent orderings and transpiler seeds')
ax.grid(alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

Most of the gain arrives early, which is the useful practical fact: a few dozen compilations capture nearly all of what a few thousand would. The curve also flattens, so there is a point past which more search buys nothing and the remaining cost is genuinely imposed by the device's connectivity.

## Does the smaller circuit give a better answer?

Fewer gates is a proxy. The question is whether the answer improves, and the two can come apart: a shallower circuit with worse-placed gates, or one whose reduction came from cancelling gates that were doing nothing, will not help. We check on a simulator with a noise model before spending credits.

In [ ]:
def cut_expectation(counts, shots):
    total = 0
    for bits, num in counts.items():
        b = bits.replace(' ', '')[::-1]
        total += num * sum(1 for u, v in EDGES if b[u] != b[v])
    return total / shots


def retained(value):
    return (value - RANDOM_GUESS) / (IDEAL_CUT - RANDOM_GUESS)


nm = NoiseModel()
nm.add_all_qubit_quantum_error(depolarizing_error(0.012, 2), ['cx'])
nm.add_all_qubit_quantum_error(depolarizing_error(0.0008, 1), ['sx', 'x', 'rz'])
noisy = AerSimulator(noise_model=nm, seed_simulator=9)

SIM_SHOTS = 20000
rows = []
for label, circ in [('single default call', default_circ), ('swept', swept_circ)]:
    c = circ.copy()
    c.measure_all()
    counts = noisy.run(c, shots=SIM_SHOTS).result().get_counts()
    val = cut_expectation(counts, SIM_SHOTS)
    rows.append({'compilation': label, 'CX': circ.count_ops().get('cx', 0),
                 'depth': circ.depth(), 'cut value': round(val, 3),
                 'advantage retained': f'{retained(val):.1%}'})

print(f"ideal {IDEAL_CUT:.3f}, random guessing {RANDOM_GUESS:.1f}")
pd.DataFrame(rows).set_index('compilation')

## On real devices

One complication: when you submit through a unified platform, the device's own compiler processes the circuit again, and a circuit we compiled ourselves can be recompiled until our work is lost.

**On Rigetti Cepheus** this is unavoidable through qBraid. We submit the two orderings as logical circuits, the platform compiles both, and the comparison shows whether our chosen ordering still helps after another compiler has processed it. This matches the usual situation: you rarely control the vendor's compiler, but you always control the circuit you give it.

**On IQM Garnet** we can do better. Garnet contains a line of eight connected qubits (14, 15, 16, 17, 12, 11, 10 and 9), the same shape as the stand-in layout we compiled for. So we send our own two compiled circuits *verbatim*: translated gate for gate into Garnet's native gates and run exactly as written. In our first tests, without this step, Garnet's compiler turned both orderings into the same circuit, so the comparison measured nothing. After every job, `check_what_ran` compares the gates we sent with the gates the device ran.

The default run is 2 jobs per device and may take minutes to hours depending on queues.

In [ ]:
provider = QbraidProvider()

# Devices are named by qBraid QRN. The README lists devices, prices and availability.
BACKENDS = {
    'Rigetti Cepheus': 'rigetti:rigetti:qpu:cepheus-1-108q',
    'IQM Garnet':      'aws:iqm:qpu:garnet',
    # 'AQT IBEX Q1':   'aws:aqt:qpu:ibex-q1',   # trapped ion; runs in scheduled windows; 2.35 credits per shot
}

COLORS = {
    'Rigetti Cepheus': '#c63792',
    'IQM Garnet':      '#1a5285',
    'AQT IBEX Q1':     '#2d7a4f',
}


# Two logically identical circuits, differing only in the order of commuting terms.
SUBMISSIONS = {
    'default order': qaoa_layer(EDGES, GAMMA, BETA),
    'swept order':   qaoa_layer(swept_order, GAMMA, BETA),
}
for qc in SUBMISSIONS.values():
    qc.measure_all()

assert Operator(qaoa_layer(EDGES, GAMMA, BETA)).equiv(
    Operator(qaoa_layer(swept_order, GAMMA, BETA))), "orderings must agree"

devices = {name: provider.get_device(dev_id) for name, dev_id in BACKENDS.items()}
print("Configured backends:", list(devices))


from qiskit import ClassicalRegister


def measured(circ):
    """Add measurements that read each logical qubit where routing left it."""
    final = circ.layout.final_index_layout() if circ.layout is not None else list(range(N_NODES))
    c = circ.copy()
    c.add_register(ClassicalRegister(N_NODES, 'c'))
    for logical, physical in enumerate(final):
        c.measure(physical, logical)
    return c


# For devices that run circuits exactly: our own compilations, unchanged.
COMPILED_SUBMISSIONS = {
    'default order': measured(default_circ),
    'swept order':   measured(swept_circ),
}
print("Submitting two circuits that implement the same unitary.")

import re

_NOT_GATES = {"openqasm", "include", "bit", "qubit", "box", "measure", "declare", "pragma",
              "b", "c", "meas", "ro", "barrier", "fence", "delay", "halt", "reset", "defcal", "cal"}

def _count_gates(program_text):
    """(all gates, two-qubit gates) in a compiled OpenQASM or Quil program."""
    total = two = 0
    for line in program_text.splitlines():
        m = re.match(r"\s*([A-Za-z_]+)", line)
        if not m or m.group(1).lower() in _NOT_GATES:
            continue
        total += 1
        if m.group(1).lower() in ("cz", "cx", "cnot", "iswap", "xy", "cphase", "ecr", "ms", "zz"):
            two += 1
    return total, two

def check_what_ran(job, circuit):
    """Compare the circuit we sent with the program the device actually ran.

    Device compilers rewrite circuits before running them. Usually that only
    changes the gate names, but a compiler can also remove gates that cancel,
    such as a circuit followed by its inverse. This prints both gate counts.
    """
    ops = [inst.operation for inst in circuit.data if inst.operation.name not in ("barrier", "measure")]
    sent_total, sent_two = len(ops), sum(1 for op in ops if op.num_qubits == 2)
    try:
        program = job.client.get_job_compiled_program(job.id)
    except Exception as err:
        print(f"  could not fetch the compiled program ({type(err).__name__}); check skipped")
        return None
    ran_total, ran_two = _count_gates(getattr(program, "data", str(program)))
    print(f"  gates sent {sent_total} ({sent_two} two-qubit); device ran {ran_total} ({ran_two} two-qubit)")
    removed = (sent_two and ran_two < 0.5 * sent_two) or (sent_total >= 10 and ran_total < 0.5 * sent_total)
    if removed:
        print("  WARNING: the compiler removed most of the gates. "
              "This result does not measure the circuit you built.")
    return not removed


# ---- Running circuits exactly as written ----------------------------------
# A device's compiler removes gates that cancel, such as a circuit followed by
# its own inverse, even across barriers. For this notebook that would erase the
# experiment. IQM Garnet (through Amazon Braket) accepts "verbatim" programs:
# native gates on named physical qubits, run gate for gate. run_exactly() uses
# that on Garnet. Rigetti has no such mode through qBraid, so there it submits
# normally, and check_what_ran() reports whether the circuit survived.
from qiskit import transpile
from qiskit.transpiler import CouplingMap
from qbraid_core.services.runtime.schemas import Program

GARNET_EDGES = [(3, 4), (3, 8), (4, 5), (8, 9), (8, 13), (9, 10), (9, 14), (10, 11), (10, 15),
                (11, 12), (11, 16), (12, 17), (13, 14), (14, 15), (14, 18), (15, 16), (15, 19),
                (16, 17), (16, 20), (18, 19), (19, 20)]
GARNET_PATH = [14, 15, 16, 17, 12, 11, 10, 9]    # eight connected qubits in a line


def runs_exactly(device_id):
    """True for devices where run_exactly() can bypass the compiler."""
    return device_id.startswith('aws:iqm:')


def garnet_verbatim(qc, physical=GARNET_PATH):
    """qc in IQM's native gates (prx, cz) on the given physical qubits, as a verbatim program."""
    phys = list(physical)[:qc.num_qubits]
    index = {p: i for i, p in enumerate(phys)}
    links = [(index[a], index[b]) for a, b in GARNET_EDGES if a in index and b in index]
    cmap = CouplingMap(links + [(b, a) for a, b in links])
    # optimization_level=0 translates gates without cancelling or merging any.
    native = transpile(qc, basis_gates=['r', 'cz'], coupling_map=cmap,
                       initial_layout=list(range(len(phys))), optimization_level=0,
                       seed_transpiler=1)
    body, reads = [], []
    for inst in native.data:
        qs = [phys[native.find_bit(q).index] for q in inst.qubits]
        name = inst.operation.name
        if name == 'r':
            theta, phi = (float(p) for p in inst.operation.params)
            body.append(f"prx({theta:.12f}, {phi:.12f}) ${qs[0]};")
        elif name == 'cz':
            body.append(f"cz ${qs[0]}, ${qs[1]};")
        elif name == 'measure':
            reads.append(f"b[{native.find_bit(inst.clbits[0]).index}] = measure ${qs[0]};")
        elif name != 'barrier':
            raise ValueError(f"unexpected gate after translation: {name}")
    return (f"OPENQASM 3.0;\nbit[{qc.num_clbits}] b;\n#pragma braket verbatim\nbox{{\n"
            + "\n".join(body) + "\n}\n" + "\n".join(reads) + "\n")


def run_exactly(name, device, qc, shots, physical=GARNET_PATH):
    """Submit qc; return (counts, ran_as_written). On Garnet it runs gate for gate."""
    if runs_exactly(BACKENDS[name]):
        job = device.submit(Program(format='qasm3', data=garnet_verbatim(qc, physical)),
                            shots=shots, tags=QUEST_JOB_TAGS)
    else:
        job = device.run(qc, shots=shots, tags=QUEST_JOB_TAGS)
    counts = job.result().data.get_counts()
    ran_as_written = check_what_ran(job, qc)
    return counts, ran_as_written is not False


In [ ]:
hw_cut = {name: {} for name in BACKENDS}

for name, device in devices.items():
    # Garnet runs our compiled circuits verbatim; elsewhere the platform compiles the orderings.
    circuits = COMPILED_SUBMISSIONS if runs_exactly(BACKENDS[name]) else SUBMISSIONS
    for label, qc in circuits.items():
        counts, _ = run_exactly(name, device, qc, HW_SHOTS)
        hw_cut[name][label] = cut_expectation(counts, HW_SHOTS)
        print(f"{name:18s} {label:14s} <cut> = {hw_cut[name][label]:.3f}   "
              f"retained {retained(hw_cut[name][label]):.1%}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

names = list(BACKENDS)
x = np.arange(len(names))
w = 0.36

for i, (label, hatch) in enumerate([('default order', ''), ('swept order', '//')]):
    vals = [retained(hw_cut[nm][label]) for nm in names]
    ax.bar(x + (i - 0.5) * w, vals, w, label=label, hatch=hatch,
           color=[COLORS[nm] for nm in names], edgecolor='white', linewidth=1.5,
           alpha=1.0 if i else 0.55)

ax.axhline(1.0, color='k', linestyle='--', linewidth=2, alpha=0.6, label='Ideal circuit')
ax.axhline(0.0, color='gray', linestyle=':', linewidth=2, label='Random guessing')
ax.set_xticks(x)
ax.set_xticklabels([nm.replace(' ', '\n') for nm in names], fontsize=9)
ax.set_ylabel('fraction of the ideal advantage retained')
ax.set_title('Same unitary, two orderings, compiled by the platform')
ax.grid(alpha=0.3, axis='y')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for name in BACKENDS:
    d, s = hw_cut[name]['default order'], hw_cut[name]['swept order']
    rows.append({
        'Backend': name,
        'Cut, default order': f'{d:.3f}',
        'Cut, swept order': f'{s:.3f}',
        'Retained, default': f'{retained(d):.1%}',
        'Retained, swept': f'{retained(s):.1%}',
        'Change': f'{retained(s) - retained(d):+.1%}',
    })

print(f"ideal {IDEAL_CUT:.3f}   random guessing {RANDOM_GUESS:.1f}")
pd.DataFrame(rows).set_index('Backend')

Read the `Change` column against the size of the effect you expected. The simulator predicted a gain of a few percentage points of retained advantage, which is real but not large, and shot noise on 2,000 shots is not negligible next to it. A single run showing a small improvement on two devices and a small regression on the third is consistent with the effect being real and with it being nothing; distinguishing those needs repetitions, which is the first exercise below.

This is the ordinary situation for compilation work. The gains are real, they are worth taking because they cost nothing, and they are small enough that claiming them requires the same care as any other measurement.

## When this stops paying off

**When the device is not limited by gate count.** The sweep minimises two-qubit gates. If the main error is idle decoherence, the quantity to minimise is duration, and a circuit with fewer gates in a longer serial chain can be worse. Minimise depth, or a weighted combination, and check which predicts the measured result.

**When the saving is not where the errors are.** Removing SWAPs between good qubits while leaving gates on a bad pair saves gate count but not error. The fix is to combine this notebook with the benchmarking notebook: rank the qubits, compile onto the good ones, then sweep.

**When the platform recompiles.** Choices made against a stand-in layout are provisional; only the ordering reaches the device. Where a real coupling map is available, use it, and the gains are larger and more predictable.

**When you over-select.** Picking the minimum of 48 compilations selects on a noisy statistic. The selected circuit really is smaller. But if you also select which device and which run to report, you are reporting the tail of a distribution.

In this notebook's sweep, the gain over the compiler's first result was modest, of the order of 10 to 25 percent in two-qubit gate count. It costs no credits, needs no new hardware, and takes one loop, which makes it the cheapest improvement in this series.

## Going further
- **Establish the error bars.** Repeat the hardware comparison five times per device, interleaving the two orderings so drift affects both equally, and decide whether the change survives.
- **Change what you minimise.** Rerun the sweep selecting on depth rather than CX count, and on estimated duration if your backend exposes gate times. Compare which selection predicts the measured cut value best.
- **Compile onto measured qubits.** Take the ranking from the benchmarking notebook, pin `initial_layout` to the best subset, then sweep. Report the gain from placement and from ordering separately.
- **Use a real topology.** Redo the sweep against an actual coupling map rather than a line, and see how much of the variance was an artifact of the stand-in.
- **Try a structured ordering.** For a complete graph the linear swap network is provably optimal and beats any random search. Implement it, and find the edge density at which it overtakes the sweep.
- **Push $p$.** At $p = 2$ and $p = 3$ the circuit doubles and triples in depth. Measure whether the sweep's percentage gain holds, and at which $p$ the retained advantage on hardware falls to zero.